# mugen walkthrough: Cargo freight dataset

This notebook walks through `mugen`, the Rust extension that finds minimal functional dependencies (FDs) over Arrow data, and compares it directly with Desbordante's Pyro on the same data.

What the notebook covers:
1. Load the data as a Polars DataFrame and look at its shape.
2. Check column cardinalities to see what kind of FDs to expect.
3. Run mugen and read its output.
4. Pass the same data as a pyarrow Table to show the input protocol.
5. Run Desbordante Pyro on the same data.
6. Compare the two FD sets by column name.
7. Time both engines, with the methodology explained.
8. Check how the gap changes with column count.
9. Check that thread count and the LHS cap behave as documented.
10. See how nulls are treated, using a small table.

The full 98-column file is not run here. Neither engine finishes it within 20 minutes; see `PIPELINE_REPORT.md`.

## Setup

Run this notebook with the `mugen (.venv)` kernel. It imports the `mugen` extension built with maturin into `.venv`, and the pinned Desbordante 2.5.0.

The setup cell searches upward from the working directory for the Cargo dataset folder, so the notebook runs from either the repository root or `harness/`.

The imports and their roles:
- `desbordante` is the C++ profiling library with Python bindings. It is the baseline.
- `mugen` is the extension under test.
- `polars` and `pyarrow` are the two columnar libraries `mugen` reads directly.
- `pandas` is needed only at the Desbordante boundary, because Desbordante 2.5 takes a pandas DataFrame.

In [1]:
import importlib.metadata as md
import time
from pathlib import Path

import desbordante
import mugen
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.csv as pacsv

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cargo+2000+freight+tracking+and+tracing').is_dir())
DATA = REPO / 'cargo+2000+freight+tracking+and+tracing'

print('desbordante', md.version('desbordante'))
print('polars', pl.__version__, '| pyarrow', pa.__version__)
print('mugen loaded from', mugen.__file__)
print('data folder', DATA)

desbordante 2.5.0
polars 2.0.0 | pyarrow 19.0.1
mugen loaded from /home/alonso/mugen/.venv/lib/python3.10/site-packages/mugen/__init__.py
data folder /home/alonso/mugen/cargo+2000+freight+tracking+and+tracing


## Step 1: Load the data

The dataset is the Cargo 2000 freight tracking and tracing log: 3,943 rows and 98 columns, mostly status and location codes. For a first walkthrough we use `c2k_subset20.csv`, the first 20 data columns. That is small enough for Desbordante to finish quickly but still has many dependencies.

We read it with Polars, the DataFrame library the mugen pipeline is built for. `pl.read_csv` infers the types and returns a columnar frame. The `nr` row counter is not in the subset. It is a sequence number, unique on every row, so it would trivially determine every other column and clutter the output.

In [2]:
path = DATA / 'c2k_subset20.csv'
df = pl.read_csv(path)
print('rows, columns:', df.shape)
print(df.columns)
df.head(3)

rows, columns: (3943, 20)
['i1_legid', 'i1_rcs_p', 'i1_rcs_e', 'i1_dep_1_p', 'i1_dep_1_e', 'i1_dep_1_place', 'i1_rcf_1_p', 'i1_rcf_1_e', 'i1_rcf_1_place', 'i1_dep_2_p', 'i1_dep_2_e', 'i1_dep_2_place', 'i1_rcf_2_p', 'i1_rcf_2_e', 'i1_rcf_2_place', 'i1_dep_3_p', 'i1_dep_3_e', 'i1_dep_3_place', 'i1_rcf_3_p', 'i1_rcf_3_e']


i1_legid,i1_rcs_p,i1_rcs_e,i1_dep_1_p,i1_dep_1_e,i1_dep_1_place,i1_rcf_1_p,i1_rcf_1_e,i1_rcf_1_place,i1_dep_2_p,i1_dep_2_e,i1_dep_2_place,i1_rcf_2_p,i1_rcf_2_e,i1_rcf_2_place,i1_dep_3_p,i1_dep_3_e,i1_dep_3_place,i1_rcf_3_p,i1_rcf_3_e
i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str
5182,199,218,210,215,609,935,736,256,"""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?"""
6523,844,584,90,297,700,1935,1415,431,"""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?"""
5878,4380,4119,90,280,456,905,547,700,"""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?""","""?"""


## Step 2: Look at column cardinalities

An FD `X -> A` holds when every pair of rows that agree on all columns in `X` also agree on `A`. Two things follow from that:
- A column with few distinct values is a likely right-hand side, since many column sets can agree on it.
- A column with one distinct value per row (a key) determines every other column.
- A column with a single value is determined by the empty set, so it appears as `{} -> column`.

Counting distinct values per column gives a rough idea of what the search will find before we run it.

In [3]:
distinct = df.select(pl.all().n_unique()).row(0, named=True)
for col, n in distinct.items():
    print(f'{col:>14}  {n:>5} distinct values')

      i1_legid   3943 distinct values
      i1_rcs_p   2272 distinct values
      i1_rcs_e   1824 distinct values
    i1_dep_1_p     78 distinct values
    i1_dep_1_e   1513 distinct values
i1_dep_1_place    140 distinct values
    i1_rcf_1_p    411 distinct values
    i1_rcf_1_e   1227 distinct values
i1_rcf_1_place     98 distinct values
    i1_dep_2_p    416 distinct values
    i1_dep_2_e    886 distinct values
i1_dep_2_place     40 distinct values
    i1_rcf_2_p    304 distinct values
    i1_rcf_2_e    751 distinct values
i1_rcf_2_place    117 distinct values
    i1_dep_3_p     25 distinct values
    i1_dep_3_e     25 distinct values
i1_dep_3_place     13 distinct values
    i1_rcf_3_p     22 distinct values
    i1_rcf_3_e     25 distinct values


## Step 3: Run mugen

`mugen` exposes one function: `discover_fds(data, max_error=0.0, max_lhs=None, threads=None)`.

- `data` is any object that implements the Arrow PyCapsule stream protocol (`__arrow_c_stream__`). Polars and pyarrow both do. `mugen` reads through that stream, so it does not depend on either library.
- `max_error=0.0` asks for exact FDs. A positive value allows approximate FDs with that error fraction. This notebook uses exact FDs.
- `max_lhs=None` puts no cap on the number of columns on the left-hand side. A number caps it (Step 9).
- `threads=None` runs the search on all cores through a rayon pool. A number builds a pool of that size.

The return value is a pair `(names, fds)`. `names` lists the column names in input order. Each FD is `(lhs, rhs)`: `lhs` is a list of column indices and `rhs` is one index. Returning indices keeps string work on the Python side. `fd_text` turns them back into readable FDs.

In [4]:
names, fds = mugen.discover_fds(df)
print(f'{len(names)} columns, {len(fds)} minimal FDs')


def fd_text(cols, fd):
    lhs, rhs = fd
    return '{' + ', '.join(cols[i] for i in lhs) + '} -> ' + cols[rhs]


for fd in fds[:5]:
    print(fd_text(names, fd))

20 columns, 946 minimal FDs
{i1_legid} -> i1_rcs_p
{i1_rcs_e, i1_dep_1_e, i1_rcf_1_e} -> i1_rcs_p
{i1_legid} -> i1_rcs_e
{i1_rcs_p, i1_dep_1_e, i1_dep_1_place, i1_rcf_1_e} -> i1_rcs_e
{i1_rcs_p, i1_dep_1_e, i1_rcf_1_p, i1_rcf_1_e} -> i1_rcs_e


## Step 4: The same data as a pyarrow Table

`mugen` does not depend on Polars. Any Arrow producer works. Here the same CSV is read with `pyarrow.csv`, which infers its own types. The FD set should not change, because it depends on the data, not on which library handed it over.

The `fd_set` helper turns an FD list into a set of `(frozenset of names, rhs name)` pairs. Comparing those sets checks the results by name, so differing column order between libraries does not matter.

In [5]:
def fd_set(cols, fd_list):
    return {(frozenset(cols[i] for i in lhs), cols[rhs]) for lhs, rhs in fd_list}


table = pacsv.read_csv(path)
names_arrow, fds_arrow = mugen.discover_fds(table)
print('polars input:', len(fds), 'FDs; pyarrow input:', len(fds_arrow), 'FDs')
print('same dependencies by name:', fd_set(names, fds) == fd_set(names_arrow, fds_arrow))

polars input: 946 FDs; pyarrow input: 946 FDs
same dependencies by name: True


## Step 5: Run Desbordante Pyro on the same data

Desbordante's Pyro is the algorithm `mugen` ports. The settings match the project harness:
- `error=0.0` for exact FDs, matching mugen's default.
- `seed=0` so Pyro's internal sampling is reproducible.
- `threads=1`, because Desbordante's default uses every core. Fixing the thread count on both sides makes this a like-for-like comparison. Step 8 looks at the all-core case.

Desbordante 2.5 accepts a pandas DataFrame directly in `load_data(table=...)`, so we convert from Polars once here. The timing step converts outside the timer, so both engines are timed from data already in memory.

In [6]:
algo = desbordante.fd.algorithms.Pyro()
algo.load_data(table=df.to_pandas())
algo.execute(error=0.0, seed=0, threads=1)
d_fds = algo.get_fds()
print(len(d_fds), 'minimal FDs from Desbordante Pyro')
for fd in d_fds[:5]:
    lhs, rhs = fd.to_name_tuple()
    print('{' + ', '.join(lhs) + '} -> ' + rhs)

946 minimal FDs from Desbordante Pyro
{i1_rcf_3_e} -> i1_dep_3_p
{i1_rcf_3_e} -> i1_dep_3_e
{i1_rcf_3_e} -> i1_dep_3_place
{i1_rcf_3_e} -> i1_rcf_3_p
{i1_dep_3_e} -> i1_dep_3_p


## Step 6: Compare the two FD sets

Both engines return minimal FDs: a set of left-hand columns determining one right-hand column. We compare by name rather than index, because the two engines number columns independently. Each left-hand side goes into a `frozenset`, since column order on the left does not matter: `{a, b} -> c` and `{b, a} -> c` are the same dependency.

If the sets are equal, the engines agree on every minimal FD. Any difference is printed below.

In [7]:
mugen_set = fd_set(names, fds)
desb_set = set()
for fd in d_fds:
    lhs, rhs = fd.to_name_tuple()
    desb_set.add((frozenset(lhs), rhs))

print('mugen:', len(mugen_set), '| desbordante:', len(desb_set))
print('identical:', mugen_set == desb_set)
print('only in mugen:', len(mugen_set - desb_set), '| only in desbordante:', len(desb_set - mugen_set))

mugen: 946 | desbordante: 946
identical: True
only in mugen: 0 | only in desbordante: 0


## Step 7: Time both engines

The timing rules:
- Each call runs 3 times and the best time is kept. The first call can include one-time costs such as allocation and lazy imports. The fastest run is the least affected by other load on the machine.
- The Desbordante timer includes `load_data`, which copies the frame into Desbordante's own structures. The pandas conversion happens outside the timer.
- The mugen timer includes the Arrow stream read, the UTF-8 cast and dictionary encoding, and the search. The Polars frame is built outside the timer. This is the cost a user pays going from a DataFrame to FDs, which is what `PIPELINE_REPORT.md` measures.
- Both run in this one process, so this step does not measure memory. The pipeline benchmark measures peak memory per process.

In [8]:
def best_of(fn, rounds=3):
    best = float('inf')
    result = None
    for _ in range(rounds):
        start = time.perf_counter()
        result = fn()
        best = min(best, time.perf_counter() - start)
    return best, result


def run_desbordante(pdf):
    algo = desbordante.fd.algorithms.Pyro()
    algo.load_data(table=pdf)
    algo.execute(error=0.0, seed=0, threads=1)
    return algo.get_fds()


pdf = df.to_pandas()
t_desb, _ = best_of(lambda: run_desbordante(pdf))
t_m1, _ = best_of(lambda: mugen.discover_fds(df, threads=1))
t_mall, _ = best_of(lambda: mugen.discover_fds(df))
print(f'desbordante (1 thread): {t_desb:.4f} s')
print(f'mugen (1 thread):      {t_m1:.4f} s  ({t_desb / t_m1:.1f}x)')
print(f'mugen (all threads):   {t_mall:.4f} s  ({t_desb / t_mall:.1f}x)')

desbordante (1 thread): 0.2245 s
mugen (1 thread):      0.0866 s  (2.6x)
mugen (all threads):   0.0141 s  (15.9x)


## Step 8: How the gap scales with width

The search space grows with the number of columns, so the gap should widen. This repeats the timing on the 14, 20, 30, and 40 column subsets. The 98-column file is left out because both engines time out there. The Desbordante 30-column run took about 2.4 s per run in the earlier pipeline benchmark, so the runtime of this cell depends mostly on those repeated runs.

Desbordante is not timed at 40 columns because its Pyro segfaults on that file, so the 40-column row shows mugen only.

The `FDs match` column compares counts only. Set-level agreement is checked in Step 6 and by `harness/validate_mugen.py`.

In [9]:
rows = []
for n in (14, 20, 30, 40):
    frame = pl.read_csv(DATA / f'c2k_subset{n}.csv')
    t_m, m_res = best_of(lambda: mugen.discover_fds(frame, threads=1))
    t_a, a_res = best_of(lambda: mugen.discover_fds(frame))
    row = {
        'columns': n,
        'FDs': len(m_res[1]),
        'FDs match': None,
        'desbordante 1t (s)': None,
        'mugen 1t (s)': round(t_m, 4),
        'mugen all (s)': round(t_a, 4),
        'speedup 1t': None,
        'speedup all': None,
    }
    # Desbordante segfaults on the 40-column file, so that row is mugen only.
    if n < 40:
        frame_pd = frame.to_pandas()
        t_d, d_res = best_of(lambda: run_desbordante(frame_pd))
        row.update({
            'FDs match': len(m_res[1]) == len(d_res),
            'desbordante 1t (s)': round(t_d, 4),
            'speedup 1t': round(t_d / t_m, 1),
            'speedup all': round(t_d / t_a, 1),
        })
    rows.append(row)

scaling = pd.DataFrame(rows)
scaling

,columns,FDs,FDs match,desbordante 1t (s),mugen 1t (s),mugen all (s),speedup 1t,speedup all
0,14,414,True,0.0729,0.0271,0.0058,2.7,12.7
1,20,946,True,0.2156,0.0839,0.0146,2.6,14.8
2,30,8250,True,2.4929,0.7722,0.1162,3.2,21.4
3,40,32242,None,NaN,5.4931,0.9596,NaN,NaN


## Step 9: Thread count and the LHS cap

Two settings change how mugen runs without changing the answer:

**Thread count.** The search runs one right-hand side per task, so the result should not depend on how many threads run those tasks. The first check confirms that for 1 thread and all threads on the 30-column subset.

**LHS cap (`max_lhs`).** Limiting the left-hand side to `k` columns is a way to bound the search when the full run is too slow. The claim is that a capped run returns exactly the minimal FDs of the uncapped run whose left-hand side has at most `k` columns. A minimal FD with `k` or fewer left-hand columns only depends on candidate sets of size `k` or fewer, so dropping larger candidates cannot remove it. The second check compares the capped output with the uncapped output filtered by size.

The helper `key` turns an FD into a hashable value. The left-hand side is sorted so that order does not matter.

In [10]:
def key(fd):
    lhs, rhs = fd
    return (tuple(sorted(lhs)), rhs)


df30 = pl.read_csv(DATA / 'c2k_subset30.csv')
names30, all_1t = mugen.discover_fds(df30, threads=1)
_, all_mt = mugen.discover_fds(df30)
print('1 thread vs all threads, same FDs:', {key(f) for f in all_1t} == {key(f) for f in all_mt})

_, capped = mugen.discover_fds(df30, max_lhs=2)
expected = [f for f in all_1t if len(f[0]) <= 2]
print(f'uncapped FDs with at most 2 LHS columns: {len(expected)}')
print(f'capped run returned: {len(capped)}')
print('capped set equals filtered uncapped set:', {key(f) for f in capped} == {key(f) for f in expected})

1 thread vs all threads, same FDs: True
uncapped FDs with at most 2 LHS columns: 779
capped run returned: 779
capped set equals filtered uncapped set: True


## Step 10: Nulls

`mugen` dictionary-encodes each column into integer codes, and nulls get their own code. So in mugen, all nulls in a column are equal to each other and different from every non-null value. Two rows that are null in the same column agree on that column.

The Cargo data has no real nulls, so none of the results above exercise this rule. Desbordante's behaviour on the DataFrame path was not verified in earlier reports. The small table below runs both engines on data with nulls, so the difference is visible rather than assumed. The expected mugen result is worked through after the output.

In [11]:
toy = pl.DataFrame({
    'city': ['Oslo', 'Oslo', None, None, 'Rome'],
    'zip': [1, 2, 3, 3, 4],
    'country': ['NO', 'NO', None, None, 'IT'],
})
print(toy)

toy_names, toy_fds = mugen.discover_fds(toy)
print('mugen:')
for fd in toy_fds:
    print('   ', fd_text(toy_names, fd))

toy_algo = desbordante.fd.algorithms.Pyro()
toy_algo.load_data(table=toy.to_pandas())
toy_algo.execute(error=0.0, seed=0, threads=1)
print('desbordante:')
for fd in toy_algo.get_fds():
    lhs, rhs = fd.to_name_tuple()
    print('    {' + ', '.join(lhs) + '} -> ' + rhs)

shape: (5, 3)
┌──────┬─────┬─────────┐
│ city ┆ zip ┆ country │
│ ---  ┆ --- ┆ ---     │
│ str  ┆ i64 ┆ str     │
╞══════╪═════╪═════════╡
│ Oslo ┆ 1   ┆ NO      │
│ Oslo ┆ 2   ┆ NO      │
│ null ┆ 3   ┆ null    │
│ null ┆ 3   ┆ null    │
│ Rome ┆ 4   ┆ IT      │
└──────┴─────┴─────────┘
mugen:
    {zip} -> city
    {country} -> city
    {city} -> country
    {zip} -> country
desbordante:
    {country} -> city
    {zip} -> city
    {zip} -> country
    {city} -> country


### Working through the nulls table

With nulls treated as one value, every column pair can be checked by hand:
- `city -> zip` fails: `Oslo` maps to both 1 and 2.
- `zip -> city` holds: 1 and 2 both map to `Oslo`, 3 maps to null in both rows, 4 maps to `Rome`.
- `zip -> country` holds for the same reason.
- `city -> country` holds: `Oslo` gives `NO`, null gives null, `Rome` gives `IT`.
- `country -> city` holds: `NO` gives `Oslo`, null gives null, `IT` gives `Rome`.
- `zip` is not a key, because the value 3 appears twice.

So mugen should report four FDs: `zip -> city`, `zip -> country`, `city -> country`, and `country -> city`. The output above shows both engines reporting that same set, which means Desbordante also treats the nulls as equal to each other. If it had treated each null as distinct, `zip -> city` and `zip -> country` would fail for `zip = 3`, because those two rows would then have different `city` and `country` values. The order of the FDs in the printout differs between the two engines; the comparison is on sets.

## What to take from this

- On this data the FD sets match Desbordante (Step 6). `harness/validate_mugen.py` checks the same across all three subsets and both input types.
- The thread count and the LHS cap behave as documented (Step 9). The cap gives the exact minimal FDs up to the given size.
- Timings are best-of-3 on one machine with other load present. Treat differences of a few milliseconds as noise. The full benchmark numbers and their caveats are in `PIPELINE_REPORT.md`.
- The 98-column file is out of reach for both engines within 20 minutes at the uncapped setting. Desbordante also did not speed up under the same cap on that file in earlier runs, so the cap is not a reliable way to make it finish.